# Overlay of our 3 Models
- Encampment growth
- Shelter bed/room pressure risks
- K-means cluster of public infrastructure, income, and population

In [ ]:
# Install necessary libraries
%pip install plotly

## Step 1: Load the Data

This cell loads the results from our three models into memory and filters them down to the most important rows.

**What each file contains:**
- `encampment_growth.csv`: A grid of points covering Toronto. Each row has a latitude, longitude, and a probability score representing how likely that area is to see encampment growth based on our model.
- `bed_based_shelters_map.csv`: Toronto shelters that operate on a per-bed capacity system, along with a pressure risk rating for each one.
- `room_based_shelters_map.csv`: Toronto shelters that operate on a per-room capacity system, also with pressure risk ratings.
- `neighbourhood_clusters.csv`: The 140 Toronto neighbourhoods labeled by the cluster our K-Means model assigned them to, based on income, population, and access to public infrastructure.

**What the filtering does:**
- `bed_high` and `room_high` keep only the shelters rated as **High** pressure risk. Lower-risk shelters are excluded to keep the map focused on the most urgent sites.
- `desert` keeps only the neighbourhoods labeled **Desert Zone (most at risk)**, which are the neighbourhoods our K-Means model identified as having the worst combination of low income, high population need, and poor access to services.

The print statements at the end confirm how many rows survived each filter so we can do a quick check before building the map.

In [10]:
import pandas as pd

enc  = pd.read_csv('OKAI-period/final model results/encampment_growth.csv')
bed  = pd.read_csv('OKAI-period/final model results/bed_based_shelters_map.csv')
room = pd.read_csv('OKAI-period/final model results/room_based_shelters_map.csv')
nbhd = pd.read_csv('neighbourhood_clusters.csv')

bed_high  = bed[bed['pressure_risk'] == 'High']
room_high = room[room['pressure_risk'] == 'High']
desert    = nbhd[nbhd['cluster_label'] == 'Desert Zone (most at risk)']

print(f"Encampment points           : {len(enc)}")
print(f"High-risk bed shelters      : {len(bed_high)}")
print(f"High-risk room shelters     : {len(room_high)}")
print(f"Desert zone neighbourhoods  : {len(desert)}")

print("enc shape    :", enc.shape,   "| columns:", list(enc.columns))
print("bed_high     :", len(bed_high), "rows")
print("room_high    :", len(room_high), "rows")
print("desert       :", len(desert), "rows")
print()
print(enc.head(3))

Encampment points           : 6786
High-risk bed shelters      : 45
High-risk room shelters     : 13
Desert zone neighbourhoods  : 36
enc shape    : (6786, 5) | columns: ['cell_id', 'lat', 'lon', 'probability', 'color_code']
bed_high     : 45 rows
room_high    : 13 rows
desert       : 36 rows

   cell_id        lat        lon  probability color_code
0    38535  43.636696 -79.344636     0.955712    #920a13
1    26173  43.657755 -79.433349     0.942452    #9f0e14
2    23428  43.664326 -79.453023     0.941419    #9f0e14


## Step 2: Build and Save the Overlay Map

This cell builds the interactive map by stacking four layers on top of each other, then saves it to a self-contained HTML file.

**Layer 1: Encampment Growth Heatmap**
We plot all 6,786 encampment grid points as a density heatmap. Each point is weighted by its probability score, so areas with higher predicted encampment growth appear in darker red. Areas with lower risk fade toward a light pink. This gives a continuous visual picture of where encampment growth pressure is concentrated across the city.

**Layer 2: Desert Zones**
The 36 neighbourhoods that our K-Means clustering model labeled as Desert Zones are shown as large semi-transparent circles centered on each neighbourhood. These are the areas with the worst scores across income, population density, and access to services. The size of the circle is set large enough to roughly cover the neighbourhood area.

**Layer 3: High-Risk Bed-Based Shelters**
The 45 bed-based shelters with a High pressure risk rating are shown as red markers. Hovering over a marker shows the shelter address and its risk score.

**Layer 4: High-Risk Room-Based Shelters**
The 13 room-based shelters with a High pressure risk rating are shown as dark red markers. These use the same hover format as the bed-based shelters.

**Saving the file**
`fig.write_html('overlay_map.html', include_plotlyjs=True)` saves the map to a file.

In [9]:
import plotly.graph_objects as go

fig = go.Figure()

# 1. Encampment growth heatmap
fig.add_trace(go.Densitymapbox(
    lat=enc['lat'],
    lon=enc['lon'],
    z=enc['probability'],
    radius=5,
    name='Encampment Growth',
    colorscale=[[0,'#fff5f0'],[0.4,'#fc8b6b'],[0.7,'#e32f27'],[1.0,'#920a13']],
    showscale=False,
    opacity=0.6
))

# 2. Desert zones
fig.add_trace(go.Scattermapbox(
    lat=desert['centroid_lat'],
    lon=desert['centroid_lon'],
    mode='markers',
    marker=dict(size=20, color="#050505", opacity=0.8),
    name='Desert Zone (K-Means)',
    text=desert['neighbourhood_name'],
    hovertemplate='<b>Desert Zone</b><br>%{text}<extra></extra>'
))

# 3. High-risk bed shelters
fig.add_trace(go.Scattermapbox(
    lat=bed_high['LATITUDE'],
    lon=bed_high['LONGITUDE'],
    mode='markers',
    marker=dict(size=10, color='red'),
    name='High-Risk Bed-Based Shelter',
    text=bed_high['LOCATION_ADDRESS'],
    customdata=bed_high['avg_future_risk'],
    hovertemplate='<b>Bed-Based Shelter (HIGH)</b><br>%{text}<br>Risk: %{customdata:.3f}<extra></extra>'
))

# 4. High-risk room shelters
fig.add_trace(go.Scattermapbox(
    lat=room_high['LATITUDE'],
    lon=room_high['LONGITUDE'],
    mode='markers',
    marker=dict(size=10, color='darkred'),
    name='High-Risk Room-Based Shelter',
    text=room_high['LOCATION_ADDRESS'],
    customdata=room_high['avg_future_risk'],
    hovertemplate='<b>Room-Based Shelter (HIGH)</b><br>%{text}<br>Risk: %{customdata:.3f}<extra></extra>'
))

fig.update_layout(
    mapbox_style='open-street-map',
    mapbox=dict(center=dict(lat=43.718, lon=-79.38), zoom=10),
    margin=dict(r=0, t=0, l=0, b=0),
    legend=dict(x=0.01, y=0.99, bgcolor='white', bordercolor='#bbb', borderwidth=1)
)

fig.write_html('overlay_map.html', include_plotlyjs=True)
print('Saved -> overlay_map.html')

Saved -> overlay_map.html


C:\Users\mahno\AppData\Local\Temp\ipykernel_4384\81594475.py:6: DeprecationWarning: *densitymapbox* is deprecated! Use *densitymap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Densitymapbox(
C:\Users\mahno\AppData\Local\Temp\ipykernel_4384\81594475.py:18: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
C:\Users\mahno\AppData\Local\Temp\ipykernel_4384\81594475.py:29: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
C:\Users\mahno\AppData\Local\Temp\ipykernel_4384\81594475.py:41: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
